In [4]:
!pip install -q -U \
langchain \
langchain-core \
langchain-community \
langchain-huggingface \
langchain-pinecone \
langchain-text-splitters \
transformers \
accelerate \
sentence-transformers \
pinecone \
unstructured \
huggingface_hub

In [5]:
import os
import textwrap

from huggingface_hub import notebook_login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFacePipeline
)

from langchain_community.document_loaders import (
    UnstructuredURLLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_pinecone import (
    PineconeVectorStore
)

from pinecone import (
    Pinecone,
    ServerlessSpec
)

from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_core.runnables import (
    RunnablePassthrough
)

from langchain_core.output_parsers import (
    StrOutputParser
)

In [6]:
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get(
    "PINECONE_API_KEY"
)

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [11]:
llm = HuggingFacePipeline(
    pipeline=pipe
)

In [12]:
urls = [
    "https://python.langchain.com/docs/introduction/",
    "https://huggingface.co/docs/transformers/index",
    "https://qwenlm.github.io/blog/"
]

In [13]:
loader = UnstructuredURLLoader(
    urls=urls
)

documents = loader.load()

In [14]:
print(
    "Documents Loaded:",
    len(documents)
)

Documents Loaded: 3


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(
    documents
)

In [16]:
print(
    "Chunks Created:",
    len(chunks)
)

Chunks Created: 15


In [17]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

In [19]:
INDEX_NAME = "website-chatbot"

In [20]:
if INDEX_NAME not in pc.list_indexes().names():

    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


In [21]:
vector_store = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings
)

In [22]:
vector_store.add_documents(
    chunks
)

['82cb94e5-bc08-4d8f-a48f-babe514a6815',
 '9685ce08-ef73-4d37-b4ef-4ae18f9eef37',
 'b6bf1bfc-3120-4ef1-a50d-abbcf02b944f',
 '2e1009cb-792a-403a-8eb6-ba712593efe2',
 'c3ad08f7-1915-43e0-97a3-769ecfa0d5a5',
 'd8ff1bc7-e42a-4620-955b-d8b887b6bd10',
 '61fb3127-5937-4d90-b1a9-58f469dd34ba',
 '534de848-1fd9-4e3e-9f6b-cd44d892db8e',
 '47bf3b13-ae17-42ee-97fe-20282f919909',
 'c813751f-c313-4266-a709-584e537c98f3',
 '25d1beda-a2aa-4747-87c5-f501783e61b2',
 'c8196062-0955-43fb-9885-58d0ac931239',
 '26c84814-b1e0-4ffa-8d96-dfc2688b45a6',
 '6c1aa4f2-196e-4ff2-805a-cd211cea5823',
 '5bc953d0-6426-40d8-bb00-88ce36df9f68']

In [23]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [24]:


template = """
You are a helpful AI assistant.
Use ONLY the provided context.
If the answer is not present in the context,
say that you do not know.

Context:
{context}

Question:
{question}

"""

prompt = ChatPromptTemplate.from_template(
    template
)

In [28]:


rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()

)

In [29]:
response= rag_chain.invoke(
    "What is LangChain?"
)
print(response)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Human: 
You are a helpful AI assistant.
Use ONLY the provided context.
If the answer is not present in the context,
say that you do not know.

Context:
[Document(id='82cb94e5-bc08-4d8f-a48f-babe514a6815', metadata={'source': 'https://python.langchain.com/docs/introduction/'}, page_content='LangChain overview\n\nLangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.\n\nAgent = Model + Harness. LangChain provides create_agent: a minimal, highly configurable harness. The harness is everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what your use case needs. Supports OpenAI, Anthropic, Google, and more.'), Document(id='c3ad08f7-1915-43e0-97a3-769ecfa0d5a5', metadata={'source': 'https://python.langchain.com/docs/introduction/'}, page_content='Built on top of LangGraph\n\nLangChain’s agent

In [33]:
while True:
  query = input("Ask a question:")
  if query.lower().strip() == "exit":
    break

  response= rag_chain.invoke(
      query
  )
  print("\nAnswer\n")
  print(textwrap.fill(response, width=100))
  print("\n")

Ask a question: exit
